In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI

In [3]:
documents = [
    "Artificial Intelligence is the simulation of human intelligence in machines.",
    "Machine Learning is a subset of AI that allows systems to learn from data.",
    "Deep Learning uses neural networks with many layers.",
    "RAG stands for Retrieval-Augmented Generation.",
    "FAISS is a library for efficient similarity search."
]

docs = [Document(page_content=text) for text in documents]

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(docs, embedding)

retriever = vectorstore.as_retriever()

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

C:\Users\mahif\AppData\Local\Temp\ipykernel_20148\898016440.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4615.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def should_retrieve(query):
    prompt = f"""
    Decide if the question requires external knowledge retrieval.

    Question: {query}

    Answer only YES or NO.
    """

    response = llm.invoke(prompt).content.strip().lower()
    
    return "yes" in response

In [5]:
def rewrite_query(query):
    prompt = f"""
    Improve the following query to make it more precise for retrieval:

    Query: {query}

    Improved Query:
    """

    response = llm.invoke(prompt).content.strip()
    return response

In [6]:
def basic_rag(query):
    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)
    return response.content

In [7]:
def agentic_rag(query):
    print(f"\n🔍 Original Query: {query}")
    
    # Step 1: Decide
    if should_retrieve(query):
        print("✅ Retrieval needed")
        
        # Step 2: Rewrite query
        improved_query = rewrite_query(query)
        print(f"✏️ Improved Query: {improved_query}")
        
        # Step 3: Retrieve + Generate
        answer = basic_rag(improved_query)
        
    else:
        print("❌ No retrieval needed")
        answer = llm.invoke(query).content
    
    return answer

In [8]:
query = "Explain RAG simply"

response = agentic_rag(query)

print("\n💡 Final Answer:\n", response)


🔍 Original Query: Explain RAG simply
❌ No retrieval needed

💡 Final Answer:
 RAG is an acronym that stands for Red, Amber, Green. It is a simple system used to visually indicate the status or progress of a project, task, or situation. 

- Red typically indicates that there are significant issues or problems that need immediate attention.
- Amber signifies that there are some concerns or risks that may impact the project or task if not addressed.
- Green indicates that everything is on track and progressing as planned.

By using the RAG system, stakeholders can quickly and easily understand the current status of a project or task, allowing them to take appropriate actions to address any issues or risks.
